# RAG Lab: Retrieval-Augmented Generation for Question Answering

## Introduction

Large language models memorize an enormous amount of world knowledge during pretraining, but that knowledge is **frozen at training time**, can be **wrong or incomplete**, and is **expensive to update** (you'd have to retrain or fine-tune the entire model). Retrieval-Augmented Generation (RAG) is one of the key "secret sauce" techniques that makes modern LLM applications actually *useful* in practice — it lets a language model consult external, up-to-date information at inference time without changing a single weight.

The idea was introduced in *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks* (Lewis et al., 2020) and has since become the default architecture for production LLM systems (chatbots with documentation access, code assistants, enterprise search, etc.).

### The RAG pipeline

RAG decomposes question-answering into two stages:

1. **Retrieval:** Given a user query $q$, find the $k$ most relevant documents (or document chunks) from an external corpus $\mathcal{C} = \{d_1, d_2, \ldots, d_N\}$.
2. **Generation:** Concatenate the retrieved documents into the prompt and let the language model generate an answer *conditioned on that context*.

Formally, if $\text{Retrieve}(q, \mathcal{C}, k) = \{d_{i_1}, \ldots, d_{i_k}\}$ returns the top-$k$ chunks, the model generates:

$$\hat{y} = \arg\max_y \; P_\theta\!\left(y \mid q, \; d_{i_1}, \ldots, d_{i_k}\right)$$

The crucial insight is that **RAG changes the system, not the model weights**. The model $\theta$ is unchanged — we simply give it better input. This is fundamentally different from fine-tuning approaches like LoRA or RLHF, which modify parameters.

### What we'll build

In this lab you will implement a minimal but fully functional RAG pipeline from scratch:

1. **Corpus construction** — build a collection of text chunks
2. **Embedding** — map text to dense vectors using a pretrained sentence encoder
3. **Indexing** — build a fast similarity-search index with FAISS
4. **Retrieval** — find the most relevant chunks for a query
5. **Generation** — prompt an LLM with and without retrieved context
6. **Comparison** — see how retrieval changes answer quality

Along the way, exercises will ask you to analyze each component and think about how it connects to production-scale RAG systems.

### Simplifications

To keep the lab tractable on a cpu, we make several simplifications relative to production RAG:

| This lab | Production RAG |
|----------|---------------|
| ~8 hand-written chunks | Millions of documents, chunked automatically |
| Single embedding model, no fine-tuning | Domain-adapted embedders, hybrid sparse+dense retrieval |
| Flat brute-force FAISS index | Approximate nearest-neighbor indices (IVF, HNSW) |
| Small instruct model (TinyLlama 1.1B) | GPT-4, Claude, Llama 70B, etc. |
| No re-ranking | Cross-encoder re-rankers, chain-of-thought verification |

## 1. The Corpus: Where Knowledge Lives

In a RAG system, the **corpus** is the external knowledge base that the model retrieves from. In production, this might be a company's documentation, a database of research papers, or the entire web. The corpus is typically split into **chunks** — contiguous passages short enough to fit in a prompt but long enough to carry useful information.

Chunking strategy matters enormously in practice: chunks that are too short lose context, chunks that are too long dilute the relevant signal with noise, and overlapping chunks can waste retrieval budget on near-duplicates.

For this lab, we use a small hand-crafted corpus of deep-learning topic summaries. Each chunk is a `dataclass` with an ID, a title, and a text body.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch

@dataclass
class Chunk:
    chunk_id: int
    title: str
    text: str

CORPUS: List[Chunk] = [
    Chunk(
        0, "GPT-3",
        "GPT-3 showed that scaling a standard autoregressive transformer can produce "
        "strong few-shot and in-context learning behavior without changing the basic "
        "next-token objective.",
    ),
    Chunk(
        1, "RLHF",
        "In RLHF, a reward model scores candidate responses and PPO is used to optimize "
        "the language model while keeping it close to a reference policy.",
    ),
    Chunk(
        2, "RAG",
        "Retrieval-augmented generation improves answers by retrieving external documents "
        "and inserting them into the prompt at inference time. RAG changes the system, "
        "not the model weights.",
    ),
    Chunk(
        3, "LoRA",
        "LoRA fine-tunes large language models by learning low-rank updates to existing "
        "weight matrices instead of updating all parameters.",
    ),
    Chunk(
        4, "Diffusion",
        "Denoising diffusion models learn to reverse a noise process. Latent diffusion "
        "reduces computational cost by running diffusion in a compressed latent space.",
    ),
    Chunk(
        5, "ViT",
        "A Vision Transformer turns an image into a sequence of patch embeddings and "
        "applies a transformer encoder over those patches.",
    ),
    Chunk(
        6, "PINNs",
        "Physics-informed neural networks incorporate differential equation residuals "
        "into the training loss so that learned solutions respect governing equations.",
    ),
    Chunk(
        7, "FNO",
        "The Fourier Neural Operator learns mappings between function spaces and applies "
        "learned transformations in Fourier space to model PDE solution operators.",
    ),
]

# Quick sanity check
for c in CORPUS:
    print(f"[{c.chunk_id}] {c.title}: {c.text[:60]}...")
print(f"\nCorpus size: {len(CORPUS)} chunks")

### Exercise 1: Thinking about chunking

**1a.** *(Conceptual)* Our chunks are 1–2 sentences each. Suppose instead we merged all 8 chunks into a single document and retrieved that every time. Why would this defeat the purpose of RAG? Think about what happens to the model's attention and the ratio of relevant-to-irrelevant tokens in the prompt.

**1b.** *(Conceptual)* Conversely, suppose we split each chunk into individual words and retrieved 2 of them. Why would that also be problematic? What information is lost?

**1c.** *(Conceptual)* In production systems, chunks are often created with **overlap** — e.g., a 512-token window sliding by 256 tokens. What failure mode does overlap address? What does it cost in terms of index size and retrieval budget?

## 2. Embedding: From Text to Vectors

To find which chunks are relevant to a query, we need a way to measure **semantic similarity** between pieces of text. Raw text is symbolic — we can't compute distances between strings in any meaningful way. The solution is to map text into a **dense vector space** where semantically similar texts land near each other.

We use a **sentence embedding model** — a neural network (typically a transformer) trained specifically so that its output vectors preserve semantic similarity. The model we use, `all-MiniLM-L6-v2`, was trained with a contrastive objective on over 1 billion sentence pairs:

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(e_q, e^+) / \tau)}{\exp(\text{sim}(e_q, e^+) / \tau) + \sum_{j} \exp(\text{sim}(e_q, e^-_j) / \tau)}$$

where $e_q$ is the query embedding, $e^+$ is a semantically similar passage, $e^-_j$ are negatives, and $\tau$ is a temperature. This is the same InfoNCE / NT-Xent loss used in CLIP and SimCLR — it pushes matching pairs together and non-matching pairs apart in embedding space.

The result: `all-MiniLM-L6-v2` maps any text to a 384-dimensional vector, and **cosine similarity** between vectors correlates with semantic similarity between texts.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBED_MODEL_NAME}")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

# Embed all corpus chunks — we prepend the title to give the embedder more context
texts = [f"{c.title}: {c.text}" for c in CORPUS]
corpus_embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
corpus_embeddings = corpus_embeddings.astype(np.float32)

print(f"Embedding matrix shape: {corpus_embeddings.shape}")
print(f"  → {corpus_embeddings.shape[0]} chunks, each a {corpus_embeddings.shape[1]}-dimensional vector")
print(f"  → Norm of first vector: {np.linalg.norm(corpus_embeddings[0]):.4f} (should be ≈1.0 since we normalized)")

### Exercise 2: Exploring the embedding space

**2a.** Compute the **full pairwise cosine similarity matrix** between all 8 corpus chunks. Since our embeddings are already L2-normalized, cosine similarity is just the dot product: `corpus_embeddings @ corpus_embeddings.T`. Display the result as a heatmap (using `matplotlib.pyplot.imshow`) with chunk titles as axis labels.

**2b.** Which pair of chunks has the highest similarity (excluding self-similarity)? Which pair has the lowest? Do these make intuitive sense given the topics?

**2c.** Embed the query `"How does retrieval-augmented generation work?"` using `embedder.encode(...)` with `normalize_embeddings=True`. Compute its cosine similarity to every corpus chunk and print the scores sorted from highest to lowest. Does the ranking match what you'd expect?

**2d.** *(Conceptual)* We prepend the title to each chunk before embedding (e.g., `"RAG: Retrieval-augmented generation improves..."`). Why might this help retrieval compared to embedding only the body text? Think about what information the title adds that might not be present in the body alone.

In [ ]:
import matplotlib.pyplot as plt

# Exercise 2 — your code here

## 3. Building the Index: Fast Similarity Search with FAISS

With 8 chunks, we could just compute cosine similarity to every chunk on every query — that's only 8 dot products. But real corpora have **millions** of chunks. Brute-force search over millions of 384-dimensional vectors for every query would be too slow for interactive use.

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search over dense vectors. It provides several index types:

| Index type | Strategy | Search time | Exact? |
|-----------|----------|-------------|--------|
| `IndexFlatIP` | Brute-force inner product | $O(Nd)$ | Yes |
| `IndexFlatL2` | Brute-force L2 distance | $O(Nd)$ | Yes |
| `IndexIVFFlat` | Inverted file with coarse quantizer | $O(Nd/k)$ amortized | No (approximate) |
| `IndexHNSW` | Hierarchical navigable small world graph | $O(d \log N)$ | No (approximate) |

We use `IndexFlatIP` (inner product) because our embeddings are L2-normalized, which means inner product equals cosine similarity:

$$\text{sim}(a, b) = \cos(a, b) = \frac{a \cdot b}{\|a\| \|b\|} = a \cdot b \quad \text{when } \|a\| = \|b\| = 1$$

For our tiny corpus, brute-force is more than fast enough. The code pattern, however, is identical to what you'd use with approximate indices at scale — you'd just swap the index type.

In [ ]:
import faiss

# Build the FAISS index
embedding_dim = corpus_embeddings.shape[1]  # 384
index = faiss.IndexFlatIP(embedding_dim)     # inner product = cosine sim for normalized vectors
index.add(corpus_embeddings)

print(f"FAISS index built:")
print(f"  → Index type: IndexFlatIP (brute-force inner product)")
print(f"  → Vectors stored: {index.ntotal}")
print(f"  → Dimensionality: {embedding_dim}")

### Exercise 3: Understanding the index

**3a.** *(Conceptual)* The `IndexFlatIP` index stores all vectors and compares every one on every query. If the corpus had 10 million chunks with 384-dimensional embeddings, how much memory would the index require (in GB) just for the vectors? (Each `float32` is 4 bytes.)

**3b.** *(Conceptual)* Approximate indices like `IndexIVFFlat` trade exactness for speed: they partition the vector space into clusters and only search a subset of clusters at query time. Why is this acceptable for RAG? Think about what happens if the second-best chunk is retrieved instead of the absolute best — does the downstream generation fail?

**3c.** *(Conceptual)* We use inner product (`IndexFlatIP`) rather than L2 distance (`IndexFlatL2`). For unit-normalized vectors, inner product and cosine similarity are the same. But what would go wrong if we forgot to normalize the embeddings and still used `IndexFlatIP`? Consider two chunks: one with a long, information-dense passage (large embedding norm) and one with a short passage (small norm).

## 4. Retrieval: Finding Relevant Chunks

With the index built, retrieval is straightforward: embed the query with the same model, search the index for the $k$ nearest neighbors, and return the corresponding chunks along with their similarity scores.

The `index.search(query_vector, k)` call returns two arrays:
- **scores** — the inner-product similarities (higher = more relevant)
- **ids** — the indices into the original corpus

This is the core of the "R" in RAG.

In [ ]:
def retrieve(query: str, chunks: List[Chunk], embedder: SentenceTransformer,
             index: faiss.IndexFlatIP, top_k: int) -> List[Tuple[Chunk, float]]:
    """
    Embed the query, search the FAISS index, and return the top-k chunks
    along with their similarity scores.
    """
    # Embed the query with the same model used for the corpus
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    q = q.astype(np.float32)

    # Search the index
    scores, ids = index.search(q, top_k)

    # Package results
    results: List[Tuple[Chunk, float]] = []
    for idx, score in zip(ids[0], scores[0]):
        results.append((chunks[int(idx)], float(score)))
    return results


# Try it out
QUERY = "What is the difference between RAG and RLHF?"

retrieved = retrieve(QUERY, CORPUS, embedder, index, top_k=2)

print(f"Query: {QUERY}\n")
print("Retrieved chunks:")
for chunk, score in retrieved:
    print(f"  score={score:.4f} | [{chunk.chunk_id}] {chunk.title}: {chunk.text}")

### Exercise 4: Retrieval behavior

**4a.** Try the following queries and retrieve the top-3 chunks for each. Print the results. Do the rankings make sense?
- `"How does fine-tuning work with low-rank matrices?"`
- `"Tell me about neural networks for physics simulations"`
- `"What is a Vision Transformer?"`

**4b.** Now try a query that is **out of domain** — something the corpus has no information about, e.g. `"What is the capital of France?"`. Retrieve the top-2 chunks. What scores do you get? Are they meaningfully lower than the scores from in-domain queries?

**4c.** *(Conceptual)* The retrieval system has no way to say "I don't know" — it will always return the $k$ nearest chunks, even if none of them are relevant. This is a fundamental limitation. What strategies could you use to detect when retrieved chunks are not actually relevant? Think about score thresholds, and what would happen if you set the threshold too high vs. too low.

**4d.** *(Conceptual)* We use the **same** embedding model for both queries and chunks. But queries and documents have very different structures — a query is a question, while a chunk is a declarative statement. Some production systems use **asymmetric** embedders (different encoders for queries and documents). Why might this help?

In [ ]:
# Exercise 4 — your code here

## 5. Generation: The Language Model

We now load a generative language model that will produce answers. The model is a standard **autoregressive transformer** — it generates text one token at a time, each token conditioned on everything before it:

$$P(y_1, y_2, \ldots, y_T) = \prod_{t=1}^{T} P_\theta(y_t \mid y_1, \ldots, y_{t-1})$$

We use HuggingFace's `AutoModelForCausalLM`, which handles the model architecture automatically. The key generation parameters are:

- **`max_new_tokens`**: caps the response length
- **`do_sample=False`**: greedy decoding (always pick the most probable token) — deterministic and reproducible for comparison purposes

The model also expects a **chat template** — a structured format with system/user/assistant roles that instruct-tuned models are trained to follow.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

GENERATOR_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading generator model: {GENERATOR_MODEL}")
print("(This may take a minute on the first run as weights are downloaded)\n")

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype="auto",
    device_map="cpu",
)

print(f"Model loaded on device: {model.device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## 6. Prompt Construction: With and Without Context

This is where the "augmented" part of RAG happens. We construct two different prompts for the same question:

1. **No-RAG (baseline):** The model receives only the question and must answer from its parametric memory — whatever it learned during pretraining.
2. **RAG:** The model receives retrieved context chunks inserted into the prompt, and is instructed to ground its answer in that context.

The prompt engineering matters. The system message tells the model to stick to the provided context, and the user message explicitly labels what is context and what is the question. Without these instructions, the model might ignore the context or hallucinate beyond it.

In [ ]:
def make_messages(question: str, context_chunks: List[Chunk] | None = None):
    """
    Build a chat-format message list for the generator model.

    Without context_chunks: a plain QA prompt (no-RAG baseline).
    With context_chunks: retrieved documents are inserted into the user message.
    """
    system = (
        "You are a helpful assistant for a deep learning course. "
        "If context is provided, answer only from that context and say "
        "when the context is insufficient."
    )

    if context_chunks:
        context = "\n\n".join(
            f"[{c.chunk_id}] {c.title}: {c.text}" for c in context_chunks
        )
        user = (
            "Use the following retrieved context to answer the question. "
            "Be concise and explicitly ground your answer in the context.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}"
        )
    else:
        user = question

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


# Let's see what the prompts look like
print("=" * 60)
print("NO-RAG PROMPT")
print("=" * 60)
no_rag_msgs = make_messages(QUERY)
for msg in no_rag_msgs:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"])

print("\n" + "=" * 60)
print("RAG PROMPT")
print("=" * 60)
retrieved_chunks = [chunk for chunk, _ in retrieved]
rag_msgs = make_messages(QUERY, context_chunks=retrieved_chunks)
for msg in rag_msgs:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"])

## 7. Generation: Comparing No-RAG vs. RAG

Now we generate answers for both prompts and compare them side by side. The `generate_answer` function applies the chat template, tokenizes, generates, and decodes — this is the standard HuggingFace generation pattern.

In [ ]:
def generate_answer(tokenizer, model, messages, max_new_tokens: int = 180) -> str:
    """
    Apply the chat template, tokenize, generate, and decode the response.
    Uses greedy decoding (do_sample=False) for reproducibility.
    """
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Strip the input tokens — we only want the newly generated ones
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


print(f"Question: {QUERY}\n")

# --- No-RAG baseline ---
no_rag_answer = generate_answer(tokenizer, model, no_rag_msgs)
print("=" * 60)
print("ANSWER WITHOUT RETRIEVAL (no-RAG baseline)")
print("=" * 60)
print(no_rag_answer)

# --- RAG answer ---
rag_answer = generate_answer(tokenizer, model, rag_msgs)
print("\n" + "=" * 60)
print("ANSWER WITH RETRIEVAL (RAG)")
print("=" * 60)
print(rag_answer)

### Exercise 5: Analyzing the comparison

**5a.** Read both answers carefully. Identify specific claims in the no-RAG answer that are:
- Correct but vague (the model "knows" something but can't be specific)
- Hallucinated (stated confidently but wrong or fabricated)
- Actually accurate

Do the same for the RAG answer. Does the RAG answer stick to the provided context? Does it hallucinate beyond it?

**5b.** *(Conceptual)* The no-RAG model must answer entirely from its **parametric memory** — information baked into the weights during pretraining. The RAG model has access to **non-parametric memory** — the retrieved chunks. Why is this distinction important for:
- **Factual accuracy**: Which memory type is more likely to be correct for specific, detailed facts?
- **Updatability**: If the information changes (e.g., a new paper supersedes an old technique), which system is easier to update?
- **Auditability**: In which system can you trace *where* an answer came from?

**5c.** *(Conceptual)* Our system message says "answer only from that context." But the model was trained on internet text and *has* parametric knowledge about these topics. What happens when the model's parametric knowledge contradicts the retrieved context? Which should win, and why might this be hard to enforce with small models?

## 8. The Effect of $k$: How Many Chunks to Retrieve

The choice of $k$ (the number of retrieved chunks) is a fundamental tradeoff in RAG:

- **Too few chunks** ($k=1$): You might miss relevant information, especially for questions that span multiple topics.
- **Too many chunks** ($k=N$, the full corpus): You've stuffed the entire corpus into the prompt, diluting the relevant signal with noise. The model must now find the needle in the haystack *inside the prompt*, which is no easier than its original task. You also burn context window space and increase latency.

The "right" $k$ depends on the corpus, the question complexity, and the model's ability to handle long contexts. Let's explore this empirically.

In [ ]:
# Generate answers for different values of k
query = "What is the difference between RAG and RLHF?"

for k in [1, 2, 4, 8]:
    retrieved_k = retrieve(query, CORPUS, embedder, index, top_k=k)
    chunks_k = [chunk for chunk, _ in retrieved_k]
    msgs = make_messages(query, context_chunks=chunks_k)
    answer = generate_answer(tokenizer, model, msgs)

    print(f"{'=' * 60}")
    print(f"k = {k} | Retrieved: {[c.title for c in chunks_k]}")
    print(f"{'=' * 60}")
    print(answer)
    print()

### Exercise 6: Exploring $k$ and query sensitivity

**6a.** Compare the answers for $k = 1, 2, 4, 8$ above. At what point does adding more chunks stop improving the answer? Does $k=8$ (the entire corpus) produce a *worse* answer than $k=2$? If so, why?

**6b.** Try the query `"What was the primary result from the GPT-3 paper?"` with $k=1$ and $k=2$. For this narrow, single-topic question, does $k=2$ add useful information or just noise?

**6c.** Now try `"How do RAG and LoRA each modify an LLM's behavior, and how are they different?"` — a question that genuinely spans two chunks. Compare $k=1$ vs $k=2$ vs $k=3$. Which $k$ gives the most complete answer?

**6d.** *(Conceptual)* In production RAG systems, the context window is a hard constraint — e.g., 4096 or 8192 tokens for many models. If each chunk is ~100 tokens, a system prompt is ~50 tokens, and the question is ~30 tokens, what is the maximum $k$ you could use with a 4096-token context window? What does this imply about chunk size in practice?

In [ ]:
# Exercise 6 — your code here

## 9. Putting It All Together: The Full RAG Pipeline

Let's wrap the entire pipeline — embed, index, retrieve, prompt, generate — into a single function so you can experiment easily with different queries.

In [ ]:
def rag_pipeline(question: str, top_k: int = 2, show_context: bool = True) -> str:
    """
    End-to-end RAG: retrieve, build prompt, generate.
    Also prints the no-RAG baseline for comparison.
    """
    # Retrieve
    results = retrieve(question, CORPUS, embedder, index, top_k)
    chunks = [chunk for chunk, _ in results]

    if show_context:
        print("Retrieved context:")
        for chunk, score in results:
            print(f"  [{chunk.chunk_id}] {chunk.title} (score={score:.3f})")
        print()

    # Generate both answers
    no_rag_answer = generate_answer(tokenizer, model, make_messages(question))
    rag_answer = generate_answer(tokenizer, model, make_messages(question, context_chunks=chunks))

    print(f"Q: {question}\n")
    print(f"--- Without RAG ---\n{no_rag_answer}\n")
    print(f"--- With RAG (k={top_k}) ---\n{rag_answer}")
    return rag_answer


# Try it
_ = rag_pipeline("What does a LoRA do?")

### Exercise 7: End-to-end experimentation

**7a.** Use `rag_pipeline` to ask at least 3 questions of your own — try a mix of:
- A question closely matching one chunk (e.g., about ViT)
- A question that requires synthesizing information from multiple chunks
- A question where the corpus is genuinely insufficient (e.g., about a topic not covered)

For each, compare the no-RAG and RAG answers. Note cases where RAG helps, where it doesn't, and where it might actually hurt.

**7b.** *(Conceptual)* We used a 1.1B parameter model (TinyLlama). Larger models like Llama 70B or GPT-4 have much stronger parametric memory — they "know" more from pretraining. Does this make RAG less valuable for large models? Consider:
- Does a larger model hallucinate less without RAG?
- Does a larger model follow the "answer from context" instruction more faithfully?
- Are there situations where even the strongest model benefits from retrieval?

**7c.** *(Conceptual)* RAG and fine-tuning (e.g., LoRA from our corpus) are two different strategies for improving a model's answers on a specific domain. Compare them along these axes:
- **Cost to update** when the underlying information changes
- **Generalization** to questions not anticipated at training time
- **Latency** at inference time
- **Privacy** — suppose the corpus contains sensitive documents

When would you choose RAG over fine-tuning, and vice versa? Are there situations where you'd use both?

In [ ]:
# Exercise 7 — your code here

## 10. Scaling Up: What Changes in Production?

The pipeline you built in this lab has the same logical structure as production RAG systems — the differences are in scale and refinement. Here's a summary of where each component would need to grow:

| Component | This lab | Production |
|-----------|----------|------------|
| **Corpus** | 8 hand-written chunks | Automated chunking of millions of documents with overlap, metadata, and versioning |
| **Embedder** | General-purpose `all-MiniLM-L6-v2` | Domain-adapted embedders, possibly fine-tuned on your data; hybrid sparse (BM25) + dense retrieval |
| **Index** | `IndexFlatIP` (brute-force) | Some sort of fast search |
| **Retrieval** | Single-stage top-$k$ | Multi-stage: fast first-pass retrieval → cross-encoder re-ranking → optional LLM-based filtering |
| **Prompt** | Simple template | Careful prompt engineering, few-shot examples, chain-of-thought, citation instructions |
| **Generator** | TinyLlama 1.1B | Large frontier models with long context windows (128K+ tokens) |
| **Evaluation** | Manual inspection | Automated metrics (faithfulness, relevance, answer correctness), human evaluation, A/B testing |

The key insight: **the architecture is the same**. Everything you built today — embedding, indexing, retrieval, prompt construction, generation — maps directly onto what runs in production. The differences are engineering and scale, not conceptual.